# Time Series Modeling with Satellite Data #

This notebook will explore the modeling and predicting soil moisture data obtained through soil stations and satellite data using ARIMA and neural networks.

https://www.tensorflow.org/tutorials/structured_data/time_series

In [1]:
import os
import datetime

import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

## Download data ##

In [2]:
folder_addr = "./merged_data/"
merged_sm_list = {}
for i in range(6):
    df = pd.read_csv(folder_addr + 'Station' + str(i + 1) + '_Merged.csv')
    merged_sm_list['Station' + str(i + 1)] = df

## Feature Engineering ##

In [3]:
day = 24 * 60 * 60
year = 365.2425 * day

for station in merged_sm_list.keys():
    df = merged_sm_list[station]
    
    dates = df.pop('Date')
    timestamp_s = pd.to_datetime(dates).map(pd.Timestamp.timestamp)
    wv = df.pop('Windspeed')
    lat = df.pop('Latitude')
    lon = df.pop('Longitude')
    wd_rad = df.pop('Winddirection') * np.pi / 180
    
    df['Wx'] = wv * np.cos(wd_rad)
    df['Wy'] = wv * np.sin(wd_rad)
    df['Year sin'] = np.sin(timestamp_s * (2 * np.pi / year))
    df['Year cos'] = np.cos(timestamp_s * (2 * np.pi / year))
    df["x_cord"] = np.cos(lat) * np.cos(lon)
    df["y_cord"] = np.sin(lat) * np.cos(lon)
    df["z_cord"] = np.sin(lon)
    merged_sm_list[station] = df

In [4]:
merged_sm_list['Station1'].head()

,Sat_SM,Ppt,Tair,RH,Srad,Wx,Wy,Year sin,Year cos,x_cord,y_cord,z_cord
0,0.224324,0.013512,15.396964,74.542381,172.204821,-1.918783,1.104877,0.968161,0.250330,-0.180165,0.291386,0.939486
1,0.191104,0.003036,17.210798,70.176905,204.373869,-2.293118,0.312779,0.972324,0.233638,-0.180165,0.291386,0.939486
2,0.210310,0.000000,18.358595,68.807440,215.361131,-2.437966,-0.611729,0.976199,0.216878,-0.180165,0.291386,0.939486
3,0.168200,0.000000,19.392440,70.173571,213.818155,-2.552956,-0.723101,0.979785,0.200053,-0.180165,0.291386,0.939486
4,0.167357,0.000000,20.166548,70.561905,221.869821,-2.623207,-0.755025,0.983081,0.183169,-0.180165,0.291386,0.939486


## Split and normalize methods ##

In [5]:
def split(df, target_col='Sat_SM', train_split=0.7, val_split = 0.2):
    n = len(df)
    train_df = df[:int(n*train_split)]
    val_df = df[int(n*train_split):int(n*(train_split + val_split))]
    test_df = df[int(n*(train_split + val_split)):]
    return (train_df, val_df, test_df)

In [6]:
def split_and_normalize(df):
    # Split the data
    train_df, val_df, test_df = split(df)
    
    # Normalize the data
    train_mean = train_df.mean()
    train_std = train_df.std()

    norm_train_df = (train_df - train_mean) / train_std
    norm_val_df = (val_df - train_mean) / train_std
    norm_test_df = (test_df - train_mean) / train_std
    
    return norm_train_df, norm_val_df, norm_test_df

## Data windowing ##

In [280]:
def generate_windows(data_df, input_width=7, shift=7, label_width=7, target_col='Sat_SM'):
    all_labels = data_df[target_col].values

    X = []
    y = []
    total_window_size = input_width + shift + label_width
    for i in range(len(data_df) - total_window_size):
        # get window based on input width
        inputs = data_df[i:i+input_width]

        # keep track of labels associated with current window
        label_start_idx = i + input_width + shift
        labels = all_labels[label_start_idx:label_start_idx+label_width]

        X.append(inputs)
        y.append(labels)

    return np.array(X), np.array(y)

In [281]:
def generate_batches(X, y, batch_size=32):
    tf_dataset = tf.data.Dataset.from_tensor_slices((X, y))
    batched_dataset = tf_dataset.repeat().batch(batch_size=batch_size, drop_remainder=True)

    # tf_dataset repeats indefinitely, need to compute number of step updates to complete 1 epoch
    steps_per_epoch = len(X) // batch_size

    return (batched_dataset, steps_per_epoch)

## Make final datasets ##

In [282]:
def make_final_datasets(df):
    train_df, val_df, test_df = split_and_normalize(df)
    
    X_train, y_train = generate_windows(train_df)
    X_val, y_val = generate_windows(val_df)
    X_test, y_test = generate_windows(test_df)

    return {
        'X_train': X_train,
        'y_train': y_train,
        'X_val': X_val,
        'y_val': y_val,
        'X_test': X_test,
        'y_test': y_test
    }

In [283]:
final_data = {}
for station, df in merged_sm_list.items():
    final_data[station] = make_final_datasets(df)

## Models ##

In [284]:
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras import losses, optimizers, regularizers
tf.get_logger().setLevel('ERROR')

In [285]:
loss_by_epoch = {}
val_performance = {}
performance = {}

station = 'Station1'

X_train = final_data[station]['X_train']
y_train = final_data[station]['y_train']

X_val = final_data[station]['X_val']
y_val = final_data[station]['y_val']

X_test = final_data[station]['X_test']
y_test = final_data[station]['y_test']

train_set, train_steps = generate_batches(X_train, y_train)
val_set, val_steps = generate_batches(X_val, y_val)
test_set, test_steps = generate_batches(X_test, y_test)

In [286]:
def train_model(model, model_name, train_set, train_steps, val_set, val_steps, criterion, optimizer, metrics, batch_size=32, nepoch=100, patience=5):
    model.compile(loss=criterion, optimizer=optimizer, metrics=metrics)
    
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, mode='min')
    ckpt = tf.keras.callbacks.ModelCheckpoint('./models/' + model_name, save_best_only=True)
    
    history = model.fit(train_set,
                        epochs=nepoch,
                        callbacks=[ckpt, early_stopping],
                        validation_data=val_set,
                        validation_steps=val_steps,
                        steps_per_epoch=train_steps,
                        verbose=0)
    
    train_loss = history.history
    val_loss = model.evaluate(val_set, steps=val_steps, batch_size=batch_size, verbose=1)
    return train_loss, val_loss

In [287]:
def test_model(model, test_set, test_steps, batch_size=32):
    test_loss = model.evaluate(test_set, steps=test_steps, batch_size=batch_size, verbose=0)
    return test_loss

### Feedback Autoregressive LSTM ###

In [312]:
class FeedBack(tf.keras.Model):
    def __init__(self, units, out_steps, num_features):
        super(FeedBack, self).__init__()
        self.out_steps = out_steps
        self.units = units
        self.lstm_cell = tf.keras.layers.LSTMCell(units)
        self.lstm_rnn = tf.keras.layers.RNN(self.lstm_cell, return_state=True)
        self.dense = models.Sequential([
            Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
            Dropout(0.5),
            BatchNormalization(),
            Dense(num_features, activation=None)
        ])
    
    def call(self, inputs, training=None):
        # Use a TensorArray to capture dynamically unrolled outputs.
        predictions = []

        # Initialize the LSTM state.
        x, *state = self.lstm_rnn(inputs)

        # This is the prediction for the first time step
        prediction = self.dense(x)

        # Insert the first prediction.
        predictions.append(prediction)

        # Run the rest of the prediction steps.
        for n in range(1, self.out_steps):
            # Use the last prediction as input.
            x = prediction
            # Execute one lstm step.
            x, state = self.lstm_cell(x, states=state,
                              training=training)
            # Convert the lstm output to a prediction.
            prediction = self.dense(x)
            # Add the prediction to the output.
            predictions.append(prediction)

        # predictions.shape => (time, batch, features)
        predictions = tf.stack(predictions)
        # predictions.shape => (batch, time, features)
        predictions = tf.transpose(predictions, [1, 0, 2])
        return predictions

In [313]:
criterion = losses.MeanSquaredError()
optimizer = optimizers.Adam(learning_rate=5e-3)
metrics = [tf.keras.metrics.MeanAbsoluteError(), tf.keras.metrics.MeanSquaredError()]

feedback_model = FeedBack(units=128, out_steps=1, num_features=1)
feedback_model_name = 'AR LSTM'

train_loss, val_loss = train_model(feedback_model, feedback_model_name, train_set, train_steps, val_set, val_steps, criterion, optimizer, metrics)
test_loss = test_model(feedback_model, test_set, test_steps)

loss_by_epoch[feedback_model_name] = train_loss
val_performance[feedback_model_name] = val_loss
performance[feedback_model_name] = test_loss

2024-05-01 16:06:39.433704: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


8/8 [==============================] - 0s 24ms/step - loss: 1.2695 - mean_absolute_error: 0.8842 - mean_squared_error: 1.2339


In [314]:
print('Val performance:', val_performance[feedback_model_name][1])
print('Test performance:', performance[feedback_model_name][1])

Val performance: 0.8841912746429443
Test performance: 1.2652443647384644


### Feedback Autoregressive BiLSTM ###

In [319]:
class BiFeedBack(tf.keras.Model):
    def __init__(self, units, out_steps, num_features):
        super(BiFeedBack, self).__init__()
        self.out_steps = out_steps
        self.units = units
        # Use a Bidirectional LSTM layer
        self.bidirectional_lstm = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(units, return_sequences=True, return_state=True))
        self.lstm_cell = tf.keras.layers.LSTMCell(units * 2)  # *2 because of concatenation of forward and backward states
        self.dense = tf.keras.layers.Dense(num_features)
    
    def call(self, inputs, training=None):
        predictions = []

        # Process the input sequence with the bidirectional LSTM.
        x, forward_h, forward_c, backward_h, backward_c = self.bidirectional_lstm(inputs)

        # Concatenate the forward and backward states
        state = [tf.keras.layers.concatenate([forward_h, backward_h]),
                 tf.keras.layers.concatenate([forward_c, backward_c])]

        # Initialize the first input of the generative process with the last output of the bidirectional LSTM
        prediction = self.dense(x[:, -1, :])

        # Insert the first prediction.
        predictions.append(prediction)

        # Run the rest of the prediction steps.
        for n in range(1, self.out_steps):
            x, state = self.lstm_cell(prediction, states=state, training=training)
            prediction = self.dense(x)
            predictions.append(prediction)

        predictions = tf.stack(predictions)
        predictions = tf.transpose(predictions, [1, 0, 2])
        return predictions

In [320]:
criterion = losses.MeanSquaredError()
optimizer = optimizers.Adam(learning_rate=1e-3)
metrics = [tf.keras.metrics.MeanAbsoluteError(), tf.keras.metrics.MeanSquaredError()]

bi_feedback_model = BiFeedBack(units=128, out_steps=1, num_features=1)
bi_feedback_model_name = 'AR BiLSTM'

train_loss, val_loss = train_model(bi_feedback_model, bi_feedback_model_name, train_set, train_steps, val_set, val_steps, criterion, optimizer, metrics)
test_loss = test_model(bi_feedback_model, test_set, test_steps)

loss_by_epoch[bi_feedback_model_name] = train_loss
val_performance[bi_feedback_model_name] = val_loss
performance[bi_feedback_model_name] = test_loss

2024-05-01 16:13:59.919561: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


8/8 [==============================] - 1s 81ms/step - loss: 0.9773 - mean_absolute_error: 0.7774 - mean_squared_error: 0.9773


In [321]:
print('Val performance:', val_performance[bi_feedback_model_name][1])
print('Test performance:', performance[bi_feedback_model_name][1])

Val performance: 0.7774174213409424
Test performance: 1.0867716073989868
